# 03 — Model leakage diagnosis

Three separate problems, measured rather than asserted.

**1. Accident-derived features are the label.** `accident_count` alone scores
AUC 0.976 — above the full model's reported 0.973. Positives come from `snapped`,
so `accident_count > 0` holds by definition: non-zero in 100% of positives against
7.7% of negatives.

**2. Time features carry zero information.** `hour`, `day_of_week` and `month` all
score AUC ≈ 0.500 with TVD < 0.006 between classes. Cause: `build_ml_dataset`
draws negative times from the positive time pool, making
`P(hour | negative) == P(hour | positive)` by construction.

**3. Uniform negative sampling teaches network composition, not risk.** 48% of
sampled negatives are `service` or `path` against 2% of positives, so "is this a
service road" is a near-perfect negative indicator. Restricting the pool to
rideable classes drops `road_only` lift from 2.81 to 1.76 — roughly half the
apparent performance was an artefact.

**Consequence for the product.** The hour input cannot change the route: the risk
ranking of road types is invariant across time windows (Spearman 1.000 morning /
midday / evening, 0.985 night), and Dijkstra compares only the ordering.

**But severity does vary by hour** — night 18.79% KSI against morning 11.70%,
intervals non-overlapping. Not because of trucks: 3.6% of morning crashes involve
trucks at 27.25% KSI, which combined with the remainder gives 11.68%, matching the
observed rate. The truck effect is real but too diluted to move the aggregate.
Time therefore belongs in the severity factor, not as a frequency feature.

---
## 1. Data

The supervised dataset produced by `build_ml_dataset`: 151,584 rows at a 3:1
negative-to-positive ratio. Positives are observed crashes snapped to OSM edges;
negatives are edge/time combinations with no recorded crash.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ml_data = pd.read_csv("data/processed/berlin_ml_risk_dataset.csv")
y = ml_data["accident_label"]

print(f"{len(ml_data):,} rows | positive rate {y.mean():.3f}")
print(ml_data["year"].value_counts().sort_index().to_string())

151,584 rows | positive rate 0.250
year
2018    20805
2019    20034
2020    20331
2021    17228
2022    18586
2023    17979
2024    17755
2025    18866


---
## 2. Leakage source 1 — accident-derived features are the label

Six columns in `NUMERIC_FEATURES` are computed from the same accidents that form
the positive labels. Measured directly: single-feature AUC, no model involved. A
column that cannot see the labels should score 0.500.

In [2]:
LEAKING = [
    "accident_count", "severity_sum", "serious_fatal_count",
    "historical_risk_norm", "node_risk_raw", "combined_spatial_risk",
]

print("Single-feature discrimination. A column that cannot see the labels")
print("should score near 0.500.\n")

for col in LEAKING:
    if col not in ml_data.columns:
        print(f"{col:24s} (absent)")
        continue
    auc = roc_auc_score(y, ml_data[col])
    pos_nz = (ml_data.loc[y == 1, col] > 0).mean()
    neg_nz = (ml_data.loc[y == 0, col] > 0).mean()
    print(f"{col:24s} AUC {auc:.3f} | non-zero: pos {pos_nz:6.1%}  neg {neg_nz:6.1%}")

Single-feature discrimination. A column that cannot see the labels
should score near 0.500.

accident_count           AUC 0.976 | non-zero: pos 100.0%  neg   7.7%
severity_sum             AUC 0.974 | non-zero: pos 100.0%  neg   7.7%
serious_fatal_count      AUC 0.656 | non-zero: pos  32.6%  neg   1.6%
historical_risk_norm     AUC 0.936 | non-zero: pos 100.0%  neg 100.0%
node_risk_raw            AUC 0.912 | non-zero: pos  93.6%  neg  16.7%
combined_spatial_risk    AUC 0.946 | non-zero: pos 100.0%  neg 100.0%


`accident_count` alone scores **0.976** — above the full model's reported 0.973.
The 100% is structural: positives come from `snapped`, so `accident_count > 0`
holds by definition, against 7.7% of randomly sampled negative edges.

This is the signature visible in `model_metrics.json` as recall of exactly 1.000
on the positive class. That is not performance, it is an identity.

---
## 3. Leakage source 2 — the time features carry no information

Less obvious, and not something we were looking for.

In [3]:
print("Negative times are drawn from the positive time pool by construction,")
print("so the marginal distributions should be near-identical — meaning the")
print("time features carry no discriminative information at all.\n")

for col in ["hour", "day_of_week", "month"]:
    p = ml_data.loc[y == 1, col].value_counts(normalize=True).sort_index()
    n = ml_data.loc[y == 0, col].value_counts(normalize=True).sort_index()
    a = pd.DataFrame({"pos": p, "neg": n}).fillna(0)
    tvd = 0.5 * (a["pos"] - a["neg"]).abs().sum()
    print(f"{col:14s} TVD {tvd:.4f} | single-feature AUC {roc_auc_score(y, ml_data[col]):.3f}")

Negative times are drawn from the positive time pool by construction,
so the marginal distributions should be near-identical — meaning the
time features carry no discriminative information at all.

hour           TVD 0.0052 | single-feature AUC 0.500
day_of_week    TVD 0.0028 | single-feature AUC 0.502
month          TVD 0.0040 | single-feature AUC 0.500


All three score AUC ≈ 0.500 with total variation distance below 0.006. The cause
is in `build_ml_dataset`:

```python
time_pool = pos[["year", "month", "hour", "day_of_week"]]
```

Negative times are drawn from the positive time pool, making
`P(hour | negative) == P(hour | positive)` by construction. Seven of the eleven
non-leaking features are time features, and all seven are noise.

The EDA finding that crashes peak Tuesday–Thursday is real — it just cannot be
learned from this dataset, because negatives peak there too.

---
## 4. Comparative retraining

Four feature sets, train 2018–2024 / test 2025. `lift` is average precision
divided by the base rate: 1.00 means no better than chance.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TIME = ["month", "hour", "day_of_week", "is_weekend", "is_rush_hour", "is_night"]
ROAD_NUM = ["edge_length_m", "has_cycleway", "maxspeed_num"]
ROAD_CAT = ["season", "highway_simple"]

FEATURE_SETS = {
    "time_only":  (TIME, []),
    "road_only":  (ROAD_NUM, ["highway_simple"]),
    "deployable": (TIME + ROAD_NUM, ROAD_CAT),
    "leaky":      (TIME + ROAD_NUM + LEAKING, ROAD_CAT),
}


def topk_recall(y_true, y_score, frac=0.10):
    k = max(1, int(len(y_true) * frac))
    top = np.argsort(-y_score)[:k]
    pos_n = np.sum(y_true)
    return float(np.sum(np.asarray(y_true)[top]) / pos_n) if pos_n else 0.0


def run(label, numeric, categorical):
    """Train on 2018-2024, test on 2025."""
    cols = numeric + categorical
    tr = ml_data["year"] < 2025
    te = ml_data["year"] == 2025

    steps = [("num", StandardScaler(), numeric)]
    if categorical:
        steps.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical))

    model = Pipeline([
        ("pre", ColumnTransformer(steps)),
        ("clf", RandomForestClassifier(
            n_estimators=300, max_depth=14, min_samples_leaf=5,
            class_weight="balanced_subsample", random_state=42, n_jobs=-1)),
    ])
    model.fit(ml_data.loc[tr, cols], ml_data.loc[tr, "accident_label"])

    y_true = ml_data.loc[te, "accident_label"].to_numpy()
    y_prob = model.predict_proba(ml_data.loc[te, cols])[:, 1]
    base = y_true.mean()
    ap = average_precision_score(y_true, y_prob)

    return {
        "features": label,
        "n": len(cols),
        "base": round(base, 3),
        "pr_auc": round(ap, 4),
        "lift": round(ap / base, 2),
        "top10_recall": round(topk_recall(y_true, y_prob), 3),
        "brier": round(brier_score_loss(y_true, y_prob), 4),
    }


results = []
for label, (num, cat) in FEATURE_SETS.items():
    num = [c for c in num if c in ml_data.columns]
    cat = [c for c in cat if c in ml_data.columns]
    results.append(run(label, num, cat))
    print(f"done: {label}")

print()
print(pd.DataFrame(results).to_string(index=False))

done: time_only
done: road_only
done: deployable
done: leaky

  features  n  base  pr_auc  lift  top10_recall  brier
 time_only  6 0.252  0.2527  1.00         0.099 0.2492
 road_only  4 0.252  0.7080  2.81         0.319 0.1422
deployable 11 0.252  0.6941  2.75         0.316 0.1444
     leaky 17 0.252  0.8877  3.52         0.362 0.0493


---
## 5. Leakage source 3 — uniform negative sampling teaches network composition

`road_only` scoring 2.81 on four features looked too good. Checking the class
distribution explains it.

In [5]:
print("Highway class distribution, positives vs negatives (%):\n")
p = ml_data.loc[y == 1, "highway_simple"].value_counts(normalize=True) * 100
n = ml_data.loc[y == 0, "highway_simple"].value_counts(normalize=True) * 100
cmp = pd.DataFrame({"positives": p, "negatives": n}).fillna(0)
cmp["ratio"] = (cmp["positives"] / cmp["negatives"].replace(0, np.nan)).round(2)
print(cmp.sort_values("positives", ascending=False).round(1).to_string())

print("\nEdge length (m):")
print(ml_data.groupby("accident_label")["edge_length_m"]
      .describe()[["25%", "50%", "75%", "mean"]].round(1).to_string())

Highway class distribution, positives vs negatives (%):

                positives  negatives  ratio
highway_simple                             
residential          32.0       30.7    1.0
secondary            28.6        5.1    5.6
tertiary             16.6        4.8    3.4
primary              14.9        1.4   10.9
cycleway              3.8        3.4    1.1
service               1.8       42.0    0.0
other                 1.1        4.9    0.2
living_street         0.8        1.6    0.5
path                  0.2        6.0    0.0
trunk                 0.0        0.0   21.0

Edge length (m):
                 25%   50%    75%  mean
accident_label                         
0               10.3  27.0   64.2  51.8
1               30.1  60.9  111.9  87.8


48% of sampled negatives are `service` or `path` against 2% of positives, so
`highway_simple == "service"` is a near-perfect negative indicator. That is
network composition, not risk — Berlin's bike graph contains 93,955 service stubs
(parking aisles, rear access) that nobody rides.

Edge length tells the same story: positive median 60.9 m against negative 27.0 m.
Accidents snap to long edges because long edges are more likely to be nearest.

---
## 6. Restricting the negative pool to the rideable network

In [6]:
# Restrict the negative pool to the network cyclists actually use. Service roads,
# tracks and paths make up 48% of sampled negatives but 2% of accidents, so
# "is it a service road" becomes a near-perfect negative indicator — an artefact
# of network composition, not a risk signal.
RIDEABLE = ["residential", "cycleway", "primary", "secondary",
            "tertiary", "living_street"]

sub = ml_data[ml_data["highway_simple"].isin(RIDEABLE)].copy()
print(f"{len(sub):,} rows | positive rate {sub['accident_label'].mean():.3f}\n")
print(sub.groupby("accident_label")["highway_simple"]
      .value_counts(normalize=True).unstack(0).mul(100).round(1).to_string())

90,136 rows | positive rate 0.407

accident_label     0     1
highway_simple            
cycleway         7.3   4.0
living_street    3.3   0.9
primary          2.9  15.4
residential     65.3  33.1
secondary       10.8  29.6
tertiary        10.3  17.2


In [7]:
full = ml_data
ml_data = sub
results_sub = []
for lbl, (num, cat) in FEATURE_SETS.items():
    num = [c for c in num if c in sub.columns]
    cat = [c for c in cat if c in sub.columns]
    results_sub.append(run(lbl, num, cat))
    print(f"done: {lbl}")
ml_data = full

print()
print(pd.DataFrame(results_sub).to_string(index=False))

done: time_only
done: road_only
done: deployable
done: leaky

  features  n  base  pr_auc  lift  top10_recall  brier
 time_only  6 0.412  0.4125  1.00         0.100 0.2519
 road_only  4 0.412  0.7228  1.76         0.203 0.1845
deployable 11 0.412  0.7049  1.71         0.199 0.1901
     leaky 17 0.412  0.8892  2.16         0.227 0.0745


`road_only` drops from 2.81 to **1.76**. Roughly half the apparent performance was
a sampling artefact.

The remaining 1.76 is still not clean: `primary` is 15.4% of positives against
2.9% of negatives, and main roads carry most of the cycling traffic. Without
exposure data, infrastructure risk and cycling volume cannot be separated.

In [8]:
# Does the risk ranking of road types change by hour? If the ordering is stable,
# a single static risk surface suffices and the hour input cannot change route
# choice. If it shifts, hour matters — but as an interaction with segment type,
# not as a standalone feature.
WINDOWS = {
    "morning 7-9":   acc["hour"].between(7, 9),
    "midday 11-14":  acc["hour"].between(11, 14),
    "evening 16-18": acc["hour"].between(16, 18),
    "night 22-4":    (acc["hour"] >= 22) | (acc["hour"] <= 4),
}

shares = {lbl: acc.loc[m, "highway_simple"].value_counts(normalize=True) * 100
          for lbl, m in WINDOWS.items()}

tab = pd.DataFrame(shares).fillna(0)
tab = tab.loc[tab.mean(axis=1).sort_values(ascending=False).index]
print(tab.round(1).to_string())

print("\nSpearman correlation between windows:")
print(tab.corr(method="spearman").round(3).to_string())

print("\nAccidents per window:")
for lbl, m in WINDOWS.items():
    print(f"  {lbl:15s} {m.sum():,}")

NameError: name 'acc' is not defined

In [ ]:
acc["is_ksi"] = acc["accident_severity"].isin([1, 2]).astype(int)

for lbl, m in WINDOWS.items():
    sel = acc[m]
    n, k = len(sel), int(sel["is_ksi"].sum())
    print(f"{lbl:15s} n={n:6,}  KSI {k/n:6.2%}")

morning 7-9     n= 7,214  KSI 11.70%
midday 11-14    n= 9,275  KSI 13.11%
evening 16-18   n= 9,333  KSI 12.46%
night 22-4      n= 1,778  KSI 18.79%


---
## 7. Conclusions

**1. Six features must be removed.** `accident_count`, `severity_sum`,
`serious_fatal_count`, `historical_risk_norm`, `node_risk_raw`,
`combined_spatial_risk`. Historical risk can return, but only built from years the
labels never touch — fit the prior on 2018–2023, label on 2024–2025, which doubles
as the temporal validation the project still lacks.

**2. The negative pool must be restricted** to classes cyclists actually use.

**3. The hour input cannot change a route.** Beyond AUC 0.500 above, the risk
ranking of road types is invariant across time windows (Spearman 1.000 morning /
midday / evening, 0.985 night — computed in `04_severity_model`). Dijkstra  compares only the ordering of edge costs,
so if the ordering does not move, the route cannot either.

**4. Report all four rows, including the leaky one, labelled as leakage.**

| Model | Features | Lift |
|---|---|---|
| Baseline | none | 1.00 |
| `time_only` | time | 1.00 |
| `road_only` | road | **1.76** |
| `deployable` | road + time | 1.71 |
| ~~`leaky`~~ | + accident-derived | 2.16 — **target leakage** |

Finding and naming a leak is a better result than a clean 0.89 nobody believes.